<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_10_xgb_model/stage_10_xgb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_09 - T2 SEQ2ONE - POST TUNING**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-15 14:18:51,490 | INFO | Environment initialized


In [42]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True  # <-- CLAVE en notebooks
)

## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-15 14:19:11,073 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Carga de datasets origen**

### **3.1. Rutas de datasets escalados**

In [39]:
# ============================================================
# ENTRADAS: T2 SPLITS
# ============================================================
IN_T2_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_T2_SUMMARY", "data/04_features/mnq_t2_summary.json"))
IN_T2_SPLITS_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_T2_SPLITS_SUMMARY", "data/05_splits/splits_summary.json"))
IN_T2_PARQUET_TRAIN = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_TRAIN", "data/05_splits/mnq_t2_train.parquet"))
IN_T2_PARQUET_VALID = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_VALID", "data/05_splits/mnq_t2_valid.parquet"))
IN_T2_PARQUET_TEST = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_TEST", "data/05_splits/mnq_t2_test.parquet"))

# ============================================================
# ENTRADAS: DATASETS ESCALADOS
# ============================================================
IN_T2_TRAIN_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_TRAIN_Z", "data/06_scaled/mnq_t2_train_z.parquet"))
IN_T2_VALID_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_VALID_Z", "data/06_scaled/mnq_t2_valid_z.parquet"))
IN_T2_TEST_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_TEST_Z", "data/06_scaled/mnq_t2_test_z.parquet"))
IN_T2_SCALER = DRIVE_DIR / Path(os.environ.get("IN_T2_SCALER", "data/06_scaled/scaler_t2.pkl"))
IN_T2_SCALER_META = DRIVE_DIR / Path(os.environ.get("IN_T2_SCALER_META", "data/06_scaled/scaler_meta_t2.json"))

### **3.2. Carga de datasets escalados**

In [43]:
# ============================================================
# CARGA DE DATASETS T2
# ============================================================

# -------------------------------
# 1. Splits originales
# -------------------------------
#df_train = pd.read_parquet(IN_T2_PARQUET_TRAIN)
#df_valid = pd.read_parquet(IN_T2_PARQUET_VALID)
#df_test  = pd.read_parquet(IN_T2_PARQUET_TEST)

#logging.info("Splits T2 cargados")
#logging.info(f"TRAIN: {df_train.shape}")
#logging.info(f"VALID: {df_valid.shape}")
#logging.info(f"TEST : {df_test.shape}")


# -------------------------------
# 2. Datasets escalados
# -------------------------------
df_train_z = pd.read_parquet(IN_T2_TRAIN_Z)
df_valid_z = pd.read_parquet(IN_T2_VALID_Z)
df_test_z  = pd.read_parquet(IN_T2_TEST_Z)

logging.info("Datasets escalados cargados")
logging.info(f"TRAIN_Z: {df_train_z.shape}")
logging.info(f"VALID_Z: {df_valid_z.shape}")
logging.info(f"TEST_Z : {df_test_z.shape}")


# -------------------------------
# 3. Scaler + metadata
# -------------------------------
import joblib

scaler_t2 = joblib.load(IN_T2_SCALER)

with open(IN_T2_SCALER_META, "r") as f:
    scaler_meta_t2 = json.load(f)

logging.info(f"Scaler cargado: {type(scaler_t2).__name__}")
logging.info(f"Scaler meta keys: {list(scaler_meta_t2.keys())}")

2026-04-15 14:45:13,076 | INFO | Datasets escalados cargados
2026-04-15 14:45:13,077 | INFO | TRAIN_Z: (462966, 18)
2026-04-15 14:45:13,079 | INFO | VALID_Z: (99134, 18)
2026-04-15 14:45:13,082 | INFO | TEST_Z : (99645, 18)
2026-04-15 14:45:13,091 | INFO | Scaler cargado: StandardScaler
2026-04-15 14:45:13,092 | INFO | Scaler meta keys: ['scaled', 'scaler_class', 'scale_cols', 'temporal_order_validated', 'sorted_by']


## **4. Carga de ventanas X/y**

### **4.1. Rutas de ventanas `seq2one` y `scaler` para L=30**

In [58]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# -------------------------------
# Validación de paths
# -------------------------------
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"[PATH] No existe: {p}")
    else:
        logger.info(f"[PATH] OK: {p}")


# ================================
# Configuración del experimento
# ================================

TARGET = "t2_dir_thr_90"   # ← simplificado (ya no lista)
WINDOW_SIZE = 30           # ← fijo
SPLITS = ["train", "valid", "test"]

FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("[CONFIG] Experimento cargado")
logger.info(f"[CONFIG] Target: {TARGET}")
logger.info(f"[CONFIG] Window size: {WINDOW_SIZE}")

2026-04-15 14:51:41,958 | INFO | [PATH] OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-15 14:51:41,959 | INFO | [PATH] OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-15 14:51:41,961 | INFO | [CONFIG] Experimento cargado
2026-04-15 14:51:41,962 | INFO | [CONFIG] Target: t2_dir_thr_90
2026-04-15 14:51:41,963 | INFO | [CONFIG] Window size: 30


In [60]:
# ================================
# Construcción de paths (simple)
# ================================

WINDOWS_L30_DIR = WINDOWS_SEQ2ONE_DIR / f"L{WINDOW_SIZE}"

TRAIN_WINDOW_PATH = WINDOWS_L30_DIR / f"windows_{TARGET}_train.npz"
VALID_WINDOW_PATH = WINDOWS_L30_DIR / f"windows_{TARGET}_valid.npz"
TEST_WINDOW_PATH  = WINDOWS_L30_DIR / f"windows_{TARGET}_test.npz"

SCALER_T2_PATH = SCALERS_DIR / "scaler_t2.pkl"

logger.info("[PATHS] Construidos")
logger.info(f"TRAIN: {TRAIN_WINDOW_PATH}")
logger.info(f"VALID: {VALID_WINDOW_PATH}")
logger.info(f"TEST : {TEST_WINDOW_PATH}")
logger.info(f"SCALER: {SCALER_T2_PATH}")


# ================================
# Validación (clara y directa)
# ================================
paths = {
    "train": TRAIN_WINDOW_PATH,
    "valid": VALID_WINDOW_PATH,
    "test": TEST_WINDOW_PATH,
    "scaler": SCALER_T2_PATH,
}

missing = [k for k, v in paths.items() if not v.exists()]

if not missing:
    logger.info("[CHECK] Todos los archivos existen")
else:
    logger.warning(f"[CHECK] Faltan: {missing}")
    for k in missing:
        logger.warning(f"{k}: {paths[k]}")

2026-04-15 14:52:43,764 | INFO | [PATHS] Construidos
2026-04-15 14:52:43,765 | INFO | TRAIN: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_train.npz
2026-04-15 14:52:43,770 | INFO | VALID: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_valid.npz
2026-04-15 14:52:43,774 | INFO | TEST : /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_test.npz
2026-04-15 14:52:43,777 | INFO | SCALER: /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-15 14:52:43,785 | INFO | [CHECK] Todos los archivos existen


### **4.2. Cargar ventanas (*.npz)**

In [61]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

In [62]:
# ================================
# Carga directa de ventanas
# ================================

logger.info("[LOAD] Ventanas L=30 | T2 thr=90")

X_train, y_train = load_npz_windows(TRAIN_WINDOW_PATH)
X_valid, y_valid = load_npz_windows(VALID_WINDOW_PATH)
X_test,  y_test  = load_npz_windows(TEST_WINDOW_PATH)

logger.info("[DONE] Ventanas cargadas")

2026-04-15 14:54:58,138 | INFO | [LOAD] Ventanas L=30 | T2 thr=90
2026-04-15 14:54:59,130 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-15 14:54:59,132 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-15 14:54:59,340 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-15 14:54:59,342 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-15 14:54:59,545 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-15 14:54:59,546 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-15 14:54:59,548 | INFO | [DONE] Ventanas cargadas


In [63]:
# ================================
# Flatten para XGBoost
# ================================

X_train = X_train.reshape(X_train.shape[0], -1)
X_valid = X_valid.reshape(X_valid.shape[0], -1)
X_test  = X_test.reshape(X_test.shape[0], -1)

logger.info("[SHAPE] Datos listos para XGBoost")
logger.info(f"TRAIN: {X_train.shape}")
logger.info(f"VALID: {X_valid.shape}")
logger.info(f"TEST : {X_test.shape}")

2026-04-15 14:55:13,628 | INFO | [SHAPE] Datos listos para XGBoost
2026-04-15 14:55:13,635 | INFO | TRAIN: (436692, 150)
2026-04-15 14:55:13,640 | INFO | VALID: (93508, 150)
2026-04-15 14:55:13,645 | INFO | TEST : (93990, 150)


### **4.3. Cargar escalador (*.pkl)**

In [65]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

In [66]:
scaler_t2 = load_scaler(SCALER_T2_PATH)

2026-04-15 14:56:52,832 | INFO | Scaler cargado: scaler_t2.pkl


## **5. Sanity Check**

Este bloque sirve para:

1. Validar que los datos están bien formados
    - Shapes correctos (X, y)
    - Sin NaN / inf
    - Consistencia entre splits
    
2. Detectar errores silenciosos
    - Targets mal alineados
    - Ventanas mal construidas
    - Clases faltantes (crítico en T2)

3. Ver distribución de clases
    - Muy importante en tu caso (clase 0 dominante)

In [72]:
def run_sanity_checks_seq2one_simple(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    tag: str = "t2_L30",
    expected_seq_len: int | None = None,
    expected_n_features: int | None = None,
    expected_flat_dim: int | None = None,
    verbose: bool = True,
):
    """
    Sanity check simplificado para seq2one.

    Soporta:
    - X 3D: (n, seq_len, n_features)
    - X 2D: (n, d_flat)

    Parámetros
    ----------
    expected_seq_len:
        Solo aplica a X 3D
    expected_n_features:
        Solo aplica a X 3D
    expected_flat_dim:
        Solo aplica a X 2D (ej: 150)
    """

    # --------------------------------------------------
    # Detectar automáticamente el modo (2D o 3D)
    # --------------------------------------------------
    X_sample = np.asarray(X_train)

    if X_sample.ndim == 3:
        mode = "3d"
    elif X_sample.ndim == 2:
        mode = "2d"
    else:
        raise ValueError(f"X_train tiene ndim inválido: {X_sample.shape}")

    # --------------------------------------------------
    # Ejecutar checks según modo
    # --------------------------------------------------
    if mode == "3d":
        kwargs = dict(
            expected_seq_len=expected_seq_len,
            expected_n_features=expected_n_features,
            expected_flat_dim=None,
        )
    else:
        kwargs = dict(
            expected_seq_len=None,
            expected_n_features=None,
            expected_flat_dim=expected_flat_dim,
        )

    out_train = sanity_check_seq2one(
        X_train, y_train,
        "train",
        **kwargs,
        verbose=verbose,
    )

    out_valid = sanity_check_seq2one(
        X_valid, y_valid,
        "valid",
        **kwargs,
        verbose=verbose,
    )

    out_test = sanity_check_seq2one(
        X_test, y_test,
        "test",
        **kwargs,
        verbose=verbose,
    )

    if verbose:
        print(f"\nOK SANITY CHECK | {tag} | mode={mode}")

    return {
        "mode": mode,
        "train": out_train,
        "valid": out_valid,
        "test": out_test,
    }

In [73]:
sanity = run_sanity_checks_seq2one_simple(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    expected_flat_dim=150,  # 30 * 5
)

[sanity_check_seq2one] train | X=(436692, 150) | y=(436692,) | mode=2d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid | X=(93508, 150) | y=(93508,) | mode=2d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test | X=(93990, 150) | y=(93990,) | mode=2d | n_classes=3 | classes=[-1, 0, 1]

OK SANITY CHECK | t2_L30 | mode=2d


## **6. Métricas de clasificación T2**

In [74]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-15 15:03:16,815 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [75]:
# ================================
# Carga de métricas (si existen)
# ================================
def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "xgb_final",
) -> pd.DataFrame:

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"[LOAD] Métricas encontradas: {path}")
        df = pd.read_parquet(path)
        logger.info(f"[LOAD] Shape: {df.shape}")
        return df

    logger.info(f"[LOAD] No existen métricas previas para: {name}")
    return pd.DataFrame()

In [76]:
# ================================
# Guardado de métricas
# ================================
def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "xgb_final",
) -> Path:

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"[SAVE] Métricas guardadas en: {out_path}")
    logger.info(f"[SAVE] Shape: {df_metrics.shape}")

    return out_path

Como usarlo:

```python
name = "xgboost_t2_L30_thr90"

df_prev = load_classification_metrics_if_exists(name=name)

if df_prev.empty:
    logger.info("No hay métricas previas → continuar entrenamiento")
else:
    logger.info("Ya existen métricas → podrías saltar entrenamiento")
```



## **8. Gestión de dispositivo y memoria**

In [77]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [78]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-15 15:05:43,488 | INFO | Seeds fijadas en 42


# **10. Modelo XGBoost**

In [ ]:
param_grid_xgb_final = {
    # ---------------------------
    # Núcleo (confirmado)
    # ---------------------------
    "n_estimators": [200],
    "max_depth": [3],
    "learning_rate": [0.03],

    # ---------------------------
    # Subsampling (robustez)
    # ---------------------------
    "subsample": [0.8],
    "colsample_bytree": [0.8],

    # ---------------------------
    # Regularización
    # ---------------------------
    "reg_lambda": [10.0],
    "reg_alpha": [0.0],

    # ---------------------------
    # Micro-ajuste (único rango abierto)
    # ---------------------------
    "min_child_weight": [1], #, 2, 3],

    # ---------------------------
    # Sin impacto
    # ---------------------------
    "gamma": [0.0],
}

## **10.1. Función `train_xgboost_final`**

In [80]:
from xgboost import XGBClassifier
import numpy as np


def train_xgboost_final(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    random_state: int = 42,
    n_jobs: int = -1,
    class_weight: str | dict | None = "balanced",
    use_gpu: bool = True,
    verbose: bool = False,
):
    """
    Modelo XGBoost final para:
    - window_size = 30
    - target = t2_dir_thr_90

    Inputs deben estar en formato 2D (flatten).
    """

    # =========================
    # 1. VALIDACIONES
    # =========================
    if X_train.ndim != 2:
        raise ValueError(f"X_train debe ser 2D (flatten). Recibido: {X_train.shape}")

    # =========================
    # 2. ENCODE LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    if set(np.unique(y_valid)) - set(classes_):
        raise ValueError("VALID contiene clases no vistas en TRAIN")

    if set(np.unique(y_test)) - set(classes_):
        raise ValueError("TEST contiene clases no vistas en TRAIN")

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 3. SAMPLE WEIGHT
    # =========================
    sample_weight = None
    weights_by_idx = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }

        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    # =========================
    # 4. MODELO (HIPERPARÁMETROS FINALES)
    # =========================
    device = "cuda" if use_gpu else "cpu"

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,

        # 🔥 PARAMS FINALES
        n_estimators=200,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        gamma=0.0,
        reg_alpha=0.0,
        reg_lambda=10.0,

        tree_method="hist",
        device=device,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 5. TRAIN
    # =========================
    model.fit(
        X_train,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 6. PREDICT VALID
    # =========================
    y_pred_valid_enc = model.predict(X_valid)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
    y_proba_valid = model.predict_proba(X_valid)

    # =========================
    # 7. PREDICT TEST
    # =========================
    y_pred_test_enc = model.predict(X_test)
    y_pred_test = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])
    y_proba_test = model.predict_proba(X_test)

    return {
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "weights_by_idx": weights_by_idx,
        "y_true_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
        "y_true_test": y_test,
        "y_pred_test": y_pred_test,
        "y_proba_test": y_proba_test,
    }

Como usarla:

```python
result = train_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)
```



## **10.2. Función `evaluate_xgboost_final`**

In [81]:
def evaluate_xgboost_final(
    *,
    result: dict,
    model_name: str = "xgboost_final",
):
    """
    Evalúa el modelo XGBoost final en VALID y TEST.

    Retorna un DataFrame con ambas evaluaciones.
    """

    rows = []

    for split in ["valid", "test"]:

        # -------------------------------
        # Selección de datos
        # -------------------------------
        y_true = result[f"y_true_{split}"]
        y_pred = result[f"y_pred_{split}"]

        # -------------------------------
        # Métricas
        # -------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target="t2_dir_thr_90",
            labels=[-1, 0, 1],
        )

        # -------------------------------
        # DataFrame
        # -------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=30,
            target="t2_dir_thr_90",
        )

        # -------------------------------
        # Metadata
        # -------------------------------
        df_row["horizon_min"] = 90
        df_row["class_weight_mode"] = "balanced"

        # Hiperparámetros finales
        df_row["n_estimators"] = 200
        df_row["max_depth"] = 3
        df_row["learning_rate"] = 0.03
        df_row["subsample"] = 0.8
        df_row["colsample_bytree"] = 0.8
        df_row["min_child_weight"] = 1
        df_row["gamma"] = 0.0
        df_row["reg_alpha"] = 0.0
        df_row["reg_lambda"] = 10.0

        rows.append(df_row)

    return pd.concat(rows, ignore_index=True)

Como usarla:

```python
result = train_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)

df_metrics = evaluate_xgboost_final(result=result)
```

## **10.3. Función `train_or_load_xgboost_final`**

In [89]:
import joblib
import json


def train_or_load_xgboost_final(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    model_name: str = "xgboost_t2_L30_thr90",
    models_dir: Path = DRIVE_DIR / "xgb_final",
    use_gpu: bool = True,
):
    """
    Entrena o carga el modelo XGBoost final, evalúa VALID/TEST
    y guarda modelo, metadata y métricas.
    """

    models_dir.mkdir(parents=True, exist_ok=True)

    model_path = models_dir / f"{model_name}.pkl"
    meta_path = models_dir / f"{model_name}_meta.json"
    metrics_path = models_dir / f"classification_{model_name}_metrics.parquet"

    meta = {
        "model_name": model_name,
        "window_size": 30,
        "target": "t2_dir_thr_90",
        "horizon_min": 90,
        "features": FEATURES_T2,
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 10.0,
        "class_weight": "balanced",
    }

    # =====================================================
    # 1. SI YA EXISTE → CARGAR
    # =====================================================
    if model_path.exists():
        logger.info(f"[LOAD] Modelo existente encontrado: {model_path}")
        model = joblib.load(model_path)

        if meta_path.exists():
            with open(meta_path, "r") as f:
                meta = json.load(f)

        df_metrics = pd.read_parquet(metrics_path) if metrics_path.exists() else pd.DataFrame()

        return {
            "model": model,
            "meta": meta,
            "df_metrics": df_metrics,
            "loaded": True,
        }

    # =====================================================
    # 2. ENTRENAR
    # =====================================================
    logger.info("[TRAIN] Entrenando modelo XGBoost final...")

    result = train_xgboost_final(
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        X_test=X_test,
        y_test=y_test,
        use_gpu=use_gpu,
    )

    model = result["model"]

    # =====================================================
    # 3. EVALUAR
    # =====================================================
    df_metrics = evaluate_xgboost_final(
        result=result,
        model_name=model_name,
    )

    # =====================================================
    # 4. GUARDAR MODELO
    # =====================================================
    joblib.dump(model, model_path)
    logger.info(f"[SAVE] Modelo guardado en: {model_path}")

    # =====================================================
    # 5. GUARDAR METADATA
    # =====================================================
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=4)

    logger.info(f"[SAVE] Metadata guardada en: {meta_path}")

    # =====================================================
    # 6. GUARDAR MÉTRICAS
    # =====================================================
    df_metrics.to_parquet(metrics_path, index=False)
    logger.info(f"[SAVE] Métricas guardadas en: {metrics_path}")

    return {
        "model": model,
        "meta": meta,
        "result": result,
        "df_metrics": df_metrics,
        "loaded": False,
    }

## **10.4. Entrenamiento de modelo XGBoost final**

In [95]:
artifact = train_or_load_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)

2026-04-15 15:24:34,529 | INFO | [TRAIN] Entrenando modelo XGBoost final...
2026-04-15 15:26:40,054 | INFO | [SAVE] Modelo guardado en: /content/drive/MyDrive/neural_profit/xgb_final/xgboost_t2_L30_thr90.pkl
2026-04-15 15:26:40,088 | INFO | [SAVE] Metadata guardada en: /content/drive/MyDrive/neural_profit/xgb_final/xgboost_t2_L30_thr90_meta.json
2026-04-15 15:26:40,184 | INFO | [SAVE] Métricas guardadas en: /content/drive/MyDrive/neural_profit/xgb_final/classification_xgboost_t2_L30_thr90_metrics.parquet


In [88]:
artifact['model']

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0.0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1, num_class=3, ...)

# **11. Métricas de modelo**

In [ ]:
import pandas as pd

# ============================================================
# 1) Copias defensivas
# ============================================================
dfs = [
    df_gru_valid.copy(),
    df_gru_test.copy(),
    df_xgb_valid.copy(),
    df_xgb_test.copy(),
]

# ============================================================
# 2) Unión de columnas
# ============================================================
all_cols = sorted(set().union(*[df.columns for df in dfs]))

dfs_aligned = [df.reindex(columns=all_cols) for df in dfs]

# ============================================================
# 3) Concatenación
# ============================================================
df_all_models = pd.concat(dfs_aligned, ignore_index=True)

# ============================================================
# 4) Orden de columnas (enfocado en comparación)
# ============================================================
preferred_cols = [
    "model",
    "family",
    "split",
    "window_size",
    "target",
    "horizon_min",
    "n_samples",
    "class_weight_mode",
    "balanced_accuracy",
    "balanced_accuracy_naive",
    "bal_acc_gain_vs_naive",
    "f1_macro",
    "f1_weighted",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "gamma",
    "reg_alpha",
    "reg_lambda",
]

remaining_cols = [c for c in df_all_models.columns if c not in preferred_cols]

df_all_models = df_all_models[
    [c for c in preferred_cols if c in df_all_models.columns] + remaining_cols
]

# ============================================================
# 5) Orden de filas
# ============================================================
df_all_models = df_all_models.sort_values(
    ["split", "model"]
).reset_index(drop=True)

# ============================================================
# 6) Resultado final
# ============================================================
display(df_all_models)

,model,family,split,window_size,target,horizon_min,n_samples,class_weight_mode,balanced_accuracy,balanced_accuracy_naive,...,recall_macro,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda
0,gru_final_test,gru_final_test,test,30,t2_dir_thr_90,90,93990,balanced,0.432091,0.333333,...,0.432091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,xgboost_final_test,xgboost_final_test,test,30,t2_dir_thr_90,90,93990,balanced,0.439816,0.333333,...,0.439816,200.0,3.0,0.03,0.8,0.8,1.0,0.0,0.0,10.0
2,gru_final,gru_final,valid,30,t2_dir_thr_90,90,93508,balanced,0.431711,0.333333,...,0.431711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,xgboost_final,xgboost_final,valid,30,t2_dir_thr_90,90,93508,balanced,0.444060,0.333333,...,0.444060,200.0,3.0,0.03,0.8,0.8,1.0,0.0,0.0,10.0


In [ ]:
df_compare = df_all_models[
    [
        "model",
        "family",
        "split",
        "window_size",
        "target",
        "balanced_accuracy",
        "bal_acc_gain_vs_naive",
        "f1_macro",
        "accuracy",
    ]
].copy()

display(df_compare)

,model,family,split,window_size,target,balanced_accuracy,bal_acc_gain_vs_naive,f1_macro,accuracy
0,gru_final_test,gru_final_test,test,30,t2_dir_thr_90,0.432091,0.098758,0.430598,0.524673
1,xgboost_final_test,xgboost_final_test,test,30,t2_dir_thr_90,0.439816,0.106482,0.439437,0.536493
2,gru_final,gru_final,valid,30,t2_dir_thr_90,0.431711,0.098378,0.428348,0.634352
3,xgboost_final,xgboost_final,valid,30,t2_dir_thr_90,0.444060,0.110727,0.445724,0.647923


In [ ]:
df_metrics = df_all_models[
    [
        "model",
        "split",
        "balanced_accuracy",
        "balanced_accuracy_naive",
        "bal_acc_gain_vs_naive",
        "f1_macro",
        "f1_weighted",
        "accuracy",
        "precision_macro",
        "recall_macro",
    ]
].copy()

display(df_metrics)

,model,split,balanced_accuracy,balanced_accuracy_naive,bal_acc_gain_vs_naive,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro
0,gru_final_test,test,0.432091,0.333333,0.098758,0.430598,0.528047,0.524673,0.434409,0.432091
1,xgboost_final_test,test,0.439816,0.333333,0.106482,0.439437,0.535982,0.536493,0.439409,0.439816
2,gru_final,valid,0.431711,0.333333,0.098378,0.428348,0.633987,0.634352,0.430275,0.431711
3,xgboost_final,valid,0.444060,0.333333,0.110727,0.445724,0.644814,0.647923,0.447728,0.444060


## **Conclusión final — Valid vs Test**



1. Consistencia entre valid y test

Ambos modelos muestran una diferencia muy pequeña entre VALID y TEST:

- GRU:
  - valid: 0.4317
  - test: 0.4321

- XGBoost:
  - valid: 0.4441
  - test: 0.4398

La caída es mínima en XGBoost (~0.004) y prácticamente nula en GRU.

Esto indica que:

- no hay sobreajuste significativo
- la señal es estable
- el pipeline está bien construido

2. Comparación final entre modelos

XGBoost sigue superando a GRU tanto en VALID como en TEST:

- Balanced accuracy (test):
  - GRU: 0.4321
  - XGBoost: 0.4398

- Gain vs naive:
  - GRU: +0.0988
  - XGBoost: +0.1065

- F1 macro:
  - GRU: 0.4306
  - XGBoost: 0.4394

La ventaja de XGBoost se mantiene fuera de muestra.

3. Interpretación técnica

El hecho de que:

- VALID ≈ TEST
- y XGBoost > GRU en ambos

implica que:

- la selección de modelo fue correcta
- no hubo sesgo en validation
- la señal capturada es real y generaliza

Además:

- GRU no logra aprovechar mejor la estructura temporal
- XGBoost captura mejor la relación entre features

4. Nivel de señal

El nivel final de señal se mantiene en:

- XGBoost: ~0.106 de gain
- GRU: ~0.099 de gain

Esto confirma que:

- la señal existe
- pero es moderada
- no hay mejoras ocultas adicionales

5. Conclusión final del proyecto

- El target T2 contiene señal real y explotable
- El modelo XGBoost es la mejor solución encontrada
- El modelo generaliza correctamente a datos no vistos
- El pipeline completo (train → valid → test) es consistente

6. Conclusión operativa

El modelo está listo para el siguiente paso:

- construcción de reglas de trading
- evaluación de rentabilidad (backtest)

No se requieren más ajustes de modelado en esta etapa.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 1) Cargar bundle (mismo dataset para ambos modelos)
# ============================================================
bundle_t2_90, _ = create_bundles(
    window_size=30,
    targets=["t2_dir_thr_90", "t2_dir_thr_120"],
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
)

bundle = bundle_t2_90

# ============================================================
# 2) Predicciones GRU y XGBoost sobre TEST
# ============================================================
preds_gru = run_gru_final_seq2one(bundle)
preds_xgb = run_xgboost_for_bundle_seq2one(bundle)

y_true = bundle["test"]["y"]
y_pred_gru = preds_gru["y_pred_test"]
y_pred_xgb = preds_xgb["y_pred_test"]

# ============================================================
# 3) Dataset base de comparación
# ============================================================
df_preds = pd.DataFrame({
    "y_true": y_true,
    "gru": y_pred_gru,
    "xgb": y_pred_xgb,
})

df_preds["agree"] = df_preds["gru"] == df_preds["xgb"]
df_preds["gru_correct"] = df_preds["gru"] == df_preds["y_true"]
df_preds["xgb_correct"] = df_preds["xgb"] == df_preds["y_true"]

# quién gana en cada fila
df_preds["winner"] = np.select(
    [
        df_preds["gru_correct"] & ~df_preds["xgb_correct"],
        ~df_preds["gru_correct"] & df_preds["xgb_correct"],
        df_preds["gru_correct"] & df_preds["xgb_correct"],
        ~df_preds["gru_correct"] & ~df_preds["xgb_correct"],
    ],
    [
        "gru_only",
        "xgb_only",
        "both_correct",
        "both_wrong",
    ],
    default="unknown",
)

# ============================================================
# 4) Resumen general
# ============================================================
summary_general = pd.DataFrame({
    "metric": [
        "n_samples",
        "gru_accuracy",
        "xgb_accuracy",
        "agreement_rate",
        "disagreement_rate",
    ],
    "value": [
        len(df_preds),
        df_preds["gru_correct"].mean(),
        df_preds["xgb_correct"].mean(),
        df_preds["agree"].mean(),
        1 - df_preds["agree"].mean(),
    ]
})

# ============================================================
# 5) Análisis de desacuerdos
# ============================================================
df_disagree = df_preds[df_preds["gru"] != df_preds["xgb"]].copy()

summary_disagree = pd.DataFrame({
    "metric": [
        "n_disagreements",
        "gru_accuracy_when_disagree",
        "xgb_accuracy_when_disagree",
        "gru_only_wins",
        "xgb_only_wins",
        "both_wrong_when_disagree",
    ],
    "value": [
        len(df_disagree),
        df_disagree["gru_correct"].mean() if len(df_disagree) > 0 else np.nan,
        df_disagree["xgb_correct"].mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "gru_only").mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "xgb_only").mean() if len(df_disagree) > 0 else np.nan,
        (df_disagree["winner"] == "both_wrong").mean() if len(df_disagree) > 0 else np.nan,
    ]
})

# ============================================================
# 6) Matriz de combinaciones GRU vs XGB
# ============================================================
combo_matrix = (
    df_preds
    .groupby(["gru", "xgb"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

# ============================================================
# 7) Precisión por coincidencia exacta entre modelos
# ============================================================
agreement_detail_rows = []

for cls in [-1, 0, 1]:
    df_cls = df_preds[(df_preds["gru"] == cls) & (df_preds["xgb"] == cls)].copy()
    agreement_detail_rows.append({
        "agreed_class": cls,
        "n_cases": len(df_cls),
        "share_of_total": len(df_cls) / len(df_preds),
        "precision_given_agreement": (df_cls["y_true"] == cls).mean() if len(df_cls) > 0 else np.nan,
    })

agreement_detail = pd.DataFrame(agreement_detail_rows)

# ============================================================
# 8) Tabla cruzada con verdad real en casos de acuerdo
# ============================================================
agree_only = df_preds[df_preds["agree"]].copy()
agree_vs_truth = (
    agree_only
    .groupby(["gru", "y_true"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

# ============================================================
# 9) Tabla cruzada con verdad real en casos de desacuerdo
# ============================================================
disagree_vs_truth = (
    df_disagree
    .groupby(["gru", "xgb", "y_true"])
    .size()
    .reset_index(name="count")
    .sort_values(["gru", "xgb", "y_true"])
    .reset_index(drop=True)
)

# ============================================================
# 10) Tabla corta de predicciones (primeras filas)
# ============================================================
preview_preds = df_preds.head(20).copy()

# ============================================================
# 11) Impresión ordenada
# ============================================================
print("=" * 90)
print("RESUMEN GENERAL")
print("=" * 90)
display(summary_general)

print("=" * 90)
print("RESUMEN DE DESACUERDOS")
print("=" * 90)
display(summary_disagree)

print("=" * 90)
print("MATRIZ DE COMBINACIONES: GRU vs XGBoost")
print("=" * 90)
display(combo_matrix)

print("=" * 90)
print("PRECISIÓN CUANDO AMBOS MODELOS COINCIDEN EN LA MISMA CLASE")
print("=" * 90)
display(agreement_detail)

print("=" * 90)
print("VERDAD REAL CUANDO AMBOS MODELOS COINCIDEN")
print("=" * 90)
display(agree_vs_truth)

print("=" * 90)
print("VERDAD REAL CUANDO LOS MODELOS DISCREPAN")
print("=" * 90)
display(disagree_vs_truth)

print("=" * 90)
print("MUESTRA DE PREDICCIONES")
print("=" * 90)
display(preview_preds)

2026-04-14 23:52:21,736 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 23:52:21,737 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 23:52:22,366 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 23:52:22,367 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 23:52:22,848 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 23:52:22,849 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 23:52:23,198 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 23:52:23,199 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 23:52:24,903 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 23:52:24,904 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 23:52:25,510 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 23:52:25,511 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 23:52:25,983 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler


KeyboardInterrupt: 

## **Interpretación detallada de la comparación GRU vs XGBoost**



1. Coincidencia entre modelos

Los dos modelos coinciden en aproximadamente el 85.1% de los casos. Esto significa que, en términos prácticos, ambos están leyendo una señal muy parecida la mayor parte del tiempo.

La consecuencia directa es que GRU y XGBoost no están aportando información completamente distinta. No parecen ser dos modelos complementarios en sentido fuerte, sino dos formas diferentes de aproximar casi la misma frontera de decisión.

2. Diferencia real entre modelos

Solo discrepan en el 14.9% de las observaciones, es decir, en 14,048 casos sobre 93,990.

Ese 14.9% es justamente la parte más interesante, porque ahí se ve si uno de los dos aporta valor adicional frente al otro.

En esos desacuerdos:

- GRU acierta en 28.8% de los casos
- XGBoost acierta en 36.7% de los casos
- ambos fallan en 34.4% de los casos

Esto muestra que, cuando los modelos piensan distinto, XGBoost resuelve mejor el conflicto. En otras palabras, en la zona difícil del problema, XGBoost toma mejores decisiones que GRU.

3. Qué está haciendo cada modelo en la práctica

La matriz de combinaciones muestra que ambos modelos predicen con mucha frecuencia la clase 0, es decir, la clase neutra:

- ambos predicen 0 al mismo tiempo en 50,865 casos
- eso representa más del 54% de toda la muestra

Además:

- ambos predicen -1 en 15,696 casos
- ambos predicen +1 en 13,381 casos

Esto confirma que ambos modelos están fuertemente alineados con la estructura del target T2: una gran parte del tiempo identifican neutralidad, y una menor proporción del tiempo identifican movimientos relevantes hacia arriba o hacia abajo.

4. Qué tan confiables son cuando coinciden

Cuando ambos coinciden en una clase, la precisión cambia bastante según la clase:

- acuerdo en -1: precisión ≈ 28.7%
- acuerdo en 0: precisión ≈ 71.6%
- acuerdo en +1: precisión ≈ 32.3%

Esto tiene una interpretación muy clara:

a. Cuando ambos dicen 0, suelen tener razón con bastante frecuencia.
Esto significa que los modelos son relativamente buenos detectando ausencia de movimiento fuerte.

b. Cuando ambos dicen -1 o +1, la precisión es bastante menor.
Esto implica que detectar movimientos significativos sigue siendo la parte difícil del problema.

En términos operativos, el modelo es mejor filtrando ruido que acertando direcciones extremas con alta pureza.

5. Implicancia para trading

Esto sugiere una interpretación muy importante:

El sistema, tal como está, no debe entenderse todavía como un generador de entradas directas de alta convicción en largo o corto. Más bien, funciona mejor como un filtro de contexto:

- identifica bastante bien cuándo no hay señal fuerte
- identifica de forma moderada cuándo podría haber una señal alcista o bajista
- pero todavía comete bastantes errores en las clases extremas

En otras palabras, el mayor valor actual del modelo parece estar en evitar operar condiciones neutras o ruidosas, más que en acertar con mucha precisión todos los movimientos fuertes.

6. Sobre el posible ensemble GRU + XGBoost

Dado que:

- la coincidencia entre ambos es muy alta
- XGBoost gana más veces cuando discrepan
- no aparece una complementariedad fuerte

la evidencia actual no sugiere que combinar ambos modelos vaya a producir una mejora importante de forma automática.

La lectura más razonable es:

- GRU no está agregando mucha señal nueva respecto a XGBoost
- XGBoost domina la comparación
- si hubiera que quedarse con un modelo base, hoy debería ser XGBoost

7. Conclusión práctica

La conclusión más sólida es la siguiente:

- ambos modelos capturan casi la misma señal
- XGBoost la captura mejor
- la principal fortaleza del sistema es detectar neutralidad
- la detección de señales alcistas y bajistas todavía es moderada, no contundente
- por lo tanto, el siguiente paso lógico no es combinar modelos, sino convertir XGBoost en reglas operativas más selectivas

8. Próximo paso más útil

Con estos resultados, lo más sólido sería analizar XGBoost por nivel de convicción, por ejemplo:

- probabilidad máxima de clase
- solo tomar casos donde predice +1 o -1 con alta confianza
- medir cómo cambia la precisión al filtrar por confianza

Ese análisis te diría si realmente puedes transformar esta señal en una regla de trading más limpia y menos ruidosa.